In [1]:
%pip install transformers accelerate hf_transfer peft -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'LGAI-EXAONE/EXAONE-4.0-1.2B'
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.bfloat16,
    device_map = 'auto' # 사용 가능한 디바이스 (CPU / GPU) 자동 설정
)

tokenizer = AutoTokenizer.from_pretrained(model_id)


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

c:\Users\SJ\OneDrive\Desktop\study\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SJ\.cache\huggingface\hub\models--LGAI-EXAONE--EXAONE-4.0-1.2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.56GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/6.70k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.91M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

In [4]:
#샘플데이터
prompt = '파이썬이 뭐야?'
chosen = '파이썬은 배우기 쉽고, 강력한 프로그래밍 언어입니다.'
rejected = '파이썬은 뱀의 일종이다.'

In [7]:
# 생성확률 계산

# 모델이 "이 답변이 얼마나 적절한지" 점수(log-prob 합)로 계산하는 함수
def get_logprob(model, tokenizer, prompt, response):
    full_text = prompt + response
    inputs = tokenizer(full_text, return_tensors='pt').to(model.device)
    prompt_len = len(tokenizer(prompt, return_tensors='pt')['input_ids'][0])

    # 학습 안하고, 계산만 진행
    with torch.no_grad():
        outputs = model(**inputs)  # 순전파
        logits = outputs.logits    # 각 위치별 다음 토큰 후보들 점수들
        log_probs = F.log_softmax(logits, dim=-1) # 점수표 -> log확률표

        token_log_probs = log_probs[:, :-1].gather(
            index = inputs['input_ids'][:, 1:].unsqueeze(-1),
            dim=-1
        ).squeeze(-1)

        response_log_probs = token_log_probs[:, prompt_len -1:]
        total = response_log_probs.sum()

    return total.item()

chosen_prob = get_logprob(model, tokenizer, prompt, chosen)
rejected_prob = get_logprob(model, tokenizer, prompt, rejected)
print(f"선호 답변 생성확률 : {chosen_prob}, 비선호 답변 생성확률 : {rejected_prob}")


선호 답변 생성확률 : -85.0, 비선호 답변 생성확률 : -52.5


log_probs[:, :-1] : seq_len의 마지막 위치는 그 다음 토큰을 맞출 정답인데 없어 길이맟춤용 마지막위치 제거          
.gather (index - ... , dim = -1): index로 준 토큰 id가 "vocab"에서 몇번째 단어인지를 의미 
/ vocab 차원에서 정답 토큰만 뽑아줌       
inputs['input_ids][:,1:]: 정답(next_token)만들기 시프트 (각 자리를 다음 토큰을 예측하기 때문에 시프트).unsqueeze(-1): gather가 요구하는 모양으로 바꾸기
token_log_probs[:, prompt_len - 1:] : 다음 토큰이 정답일 log 확률

로그호가률 합은 기본적으로 음수로 나오는데, 이 중 큰 값(0에 가까운 값)일수록 더 높은 확률

In [8]:
# dpo 손실함수 : 정책모델, 참조모델의 차이를 작게 만드는 목적의 손실함수
# (정책모델이 선호답변에 준 log-prob점수, ...비선호 답변..., 참조 모델이 선호답변에 준 log_prob점수, ...비선호 답변...)
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta = 0.1):
    logits = beta * ((policy_chosen - policy_rejected) - (ref_chosen - ref_rejected))
    loss = -F.logsigmoid(torch.tensor(logits))
    return loss.item() # float 반환